# 003b - Local Indexing batch

Written by Jean-Baptiste Jacob

Last updated: 18/02/2026

Run local_indexing.py to perform local indexing for a series of datasets. Parameters for local indexing are edited directly in the script file.

In [1]:
import sys

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess


if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)

# Setting path via: 
sys.path.insert(0, /home/esrf/jean1994b/ImageD11 )
# Running from: /home/esrf/jean1994b/ImageD11/ImageD11/__init__.py


In [2]:
import os
import matplotlib.pyplot as pl
import numpy as np

import ImageD11.sinograms.dataset, ImageD11.columnfile
    
from pf3dxrd.pf3dxrd import utils, pixelmap
from pf3dxrd.scripts import local_indexing
from pf3dxrd.sandbox import reindex_quartz_trigonal

%matplotlib ipympl
%load_ext autoreload
%autoreload 2

In [5]:
#phase to index
pname = 'garnet'

# paths
#data_dir = '/home/esrf/jean1994b/test_datasets/SI3/'            # folder containing your data
#parfile = os.path.join(data_dir,'SI3.par')                      # parameter file

data_dir = '/home/esrf/jean1994b/test_datasets/C18A_01_z180/'
parfile= data_dir+'/C18A_01.par'

indexing_pars = os.path.join(data_dir, f'indexing_pars_{pname}.json')    # indexing pars

# list of datasets to process
#dset_list = ['SI3_DT360Z5480'] 
dset_list = ['C18A_01_z180']

#### Check indexing options
have a look at indexing options and update them if needed. If 003_local_indexing_test was run beforehand, they should be saved in a json file. Otherwise, they can be defined and saved here.
OPTS.unitcell is not saved but re-defined directly in the main() function

In [8]:
OPTS = local_indexing.Options.load(indexing_pars)
#OPTS.save(indexing_pars)
print(OPTS)

[LOADED] Options loaded from /home/esrf/jean1994b/test_datasets/C18A_01_z180/indexing_pars_garnet.json
hkltol1: 0.1
hkltol2: 0.03
minpks: 10
maxpks: 5000
minpks_prop: 0.1
nrings: 10
max_mult: 24
px_kernel_size: 3
chunksize: 200
symmetry: cubic
unitcell: None
ncpu: 9


### prepare batch command and run

In [9]:
# functions to get path names and return full command from dataset name. may need to be edited depending on your file structure and naming convention
def get_full_path(dsname, stem):
    return os.path.join(data_dir, f'{dsname}_{stem}')


def get_command(dsname):
    pksfile  = get_full_path(dsname, 'pk2d_p_flt.h5')
    dsfile   = get_full_path(dsname, 'dataset.h5')
    xmapfile = get_full_path(dsname, 'xmap.h5')
    
    indpars  = os.path.join(data_dir,indexing_pars)
    
    command = f'{local_indexing.__file__} -pksfile {pksfile} -dsfile {dsfile} -xmapfile {xmapfile}'\
    f' -parfile {parfile} -pname {pname} -indexing_pars {indpars}' 
    return command

In [10]:
# job list
jobs = [get_command(dsname) for dsname in dset_list]

print(jobs[0])
len(jobs)

/home/esrf/jean1994b/pf3dxrd/scripts/local_indexing.py -pksfile /home/esrf/jean1994b/test_datasets/C18A_01_z180/C18A_01_z180_pk2d_p_flt.h5 -dsfile /home/esrf/jean1994b/test_datasets/C18A_01_z180/C18A_01_z180_dataset.h5 -xmapfile /home/esrf/jean1994b/test_datasets/C18A_01_z180/C18A_01_z180_xmap.h5 -parfile /home/esrf/jean1994b/test_datasets/C18A_01_z180//C18A_01.par -pname garnet -indexing_pars /home/esrf/jean1994b/test_datasets/C18A_01_z180/indexing_pars_garnet.json


1

In [11]:
# submit
for j in jobs:
    !python {j}



[LOG] Writing output to /home/esrf/jean1994b/test_datasets/C18A_01_z180/local_indexing_20260219_135347.log


=============================-
load data...

Pixelmap:
 size: (110, 110),
 phases: ['notIndexed', 'quartz', 'garnet'],
 phase_ids: [-1, 0, 1],
 titles: ['xyi', 'xi', 'yi', 'phase_id', 'grain_id', 'Npks', 'U', 'UBI', 'completeness', 'drlv2', 'indx_completeness', 'nindx', 'phase_label_confidence', 'uniqueness', 'unitcell'], 
 grains: 0
Reading your columnfile in hdf format
sorting peakfile by friedel pair index...
Total size =  860.67 MB
loading options from indexing_pars_garnet.json
[LOADED] Options loaded from /home/esrf/jean1994b/test_datasets/C18A_01_z180/indexing_pars_garnet.json
updated symmetry to cubic. Symmetry group defined in self.sym.group

---------------------------------
PARAMETERS FOR INDEXING:
hkltol1: 0.1
hkltol2: 0.03
minpks: 10
maxpks: 5000
minpks_prop: 0.1
nrings: 10
max_mult: 24
px_kernel_size: 3
chunksize: 200
symmetry: cubic
unitcell: Unitcell | [11.645 11

### Quartz re-indexing (Trigonal)
Run `reindex_quartz_trigonal.py` on pre-indexed quartz map. Uses structure factors to solve Dauphiné Twinning structure in quartz, by comparing trigonal hkls with high intensity contrast. 

In [12]:
def get_command_bis(dsname):
    pksfile  = get_full_path(dsname, 'pk2d_p_flt.h5')
    dsfile   = get_full_path(dsname, 'dataset.h5')
    xmapfile = get_full_path(dsname, 'xmap.h5')
    
    command = f'{reindex_quartz_trigonal.__file__} -pksfile {pksfile} -dsfile {dsfile} -xmapfile {xmapfile} -parfile {parfile}' 
    return command

In [13]:
# job list
jobs = [get_command_bis(dsname) for dsname in dset_list]

print(jobs[0])
len(jobs)

/home/esrf/jean1994b/pf3dxrd/sandbox/reindex_quartz_trigonal.py -pksfile /home/esrf/jean1994b/test_datasets/SI3/SI3_DT360Z5480_pk2d_p_flt.h5 -dsfile /home/esrf/jean1994b/test_datasets/SI3/SI3_DT360Z5480_dataset.h5 -xmapfile /home/esrf/jean1994b/test_datasets/SI3/SI3_DT360Z5480_xmap.h5 -parfile /home/esrf/jean1994b/test_datasets/SI3/SI3.par


1

In [7]:
!python {jobs[0]}


[LOG] Writing output to /home/esrf/jean1994b/test_datasets/SI3/reindex_quartz_trigonal_20251202_125204.log


=============================-
loading data...

phases: ['notIndexed', 'quartz', 'magnetite', 'biotite', 'orthoclase', 'oligoclase']
Reading your columnfile in hdf format
Total size =  36.68 MB

---------------------------------
Options:
ncpu        : 39
chunksize   : 200
kernel_size : 3
hkl_tol     : 0.1
hkl_strong  : ((2, 0, 3), (0, 3, 1))
hkl_weak    : ((0, 2, 3), (3, 0, 1))
flipmat     :
 [[-1  0  0]
 [ 0 -1  0]
 [ 0  0  1]]
---------------------------------
Number of pixels to process: 1495

Computing hkl intensities
Computing peak intensities for following hkl groups: ((2, 0, 3), (0, 3, 1))
100%|##########| 1495/1495 [00:02<00:00, 548.19it/s]
Computing peak intensities for following hkl groups: ((0, 2, 3), (3, 0, 1))
100%|##########| 1495/1495 [00:02<00:00, 591.52it/s]

738 wrongly indexed pixels found, representing 49.36% of quartz pixels.
correcting bad pixels...

Make 

In [14]:
dsname = dset_list[0]

pksfile  = get_full_path(dsname, 'pk2d_p_flt.h5')
dsfile   = get_full_path(dsname, 'dataset.h5')
xmapfile = get_full_path(dsname, 'xmap.h5')

In [29]:
OPTS = reindex_quartz_trigonal.Options()
OPTS.display()

Options:
ncpu        : 39
chunksize   : 500
kernel_size : 3
hkl_tol     : 0.1
hkl_strong  : ((2, 0, 3), (0, 3, 1))
hkl_weak    : ((0, 2, 3), (3, 0, 1))
flipmat     :
 [[-1  0  0]
 [ 0 -1  0]
 [ 0  0  1]]


In [16]:
xmap, cf = reindex_quartz_trigonal.load_data(pksfile, xmapfile, dsfile, parfile)

phases: ['notIndexed', 'quartz', 'magnetite', 'biotite', 'orthoclase', 'oligoclase']
Reading your columnfile in hdf format
Total size =  36.68 MB


In [31]:
out = reindex_quartz_trigonal.get_hkl_ints_parallel(cf, xmap, hkl=OPTS.hkl_strong, UseDefaultUBI=True)

Computing peak intensities for following hkl groups: ((2, 0, 3), (0, 3, 1))
Number of pixels to process: 1497


  0%|          | 1/1497 [00:01<45:36,  1.83s/it]

px 740041: orientation matrix U is not unitary, np.dot(U.T, U)!=np.eye(3,3)


 29%|██▉       | 441/1497 [00:01<00:03, 317.04it/s]

px 790092: orientation matrix U is not unitary, np.dot(U.T, U)!=np.eye(3,3)


100%|██████████| 1497/1497 [00:01<00:00, 756.39it/s]
